# Synthetic Organizational Document Generation for Brazilian ESG Companies

This notebook implements a synthetic data generation pipeline combining concepts from:

- Survey Paper: "On LLMs-Driven Synthetic Data Generation, Curation, and Evaluation" (Lin Long et al.)
- DocGenie Paper: "DocGenie: A Framework for High-Fidelity Synthetic Document Generation" (Harikrishnan P M et al.)

## 1. Imports and Setup

In [30]:
import os
import json
import requests
import random
import csv
from io import StringIO
from typing import List, Dict, Any
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

from dotenv import load_dotenv
load_dotenv()

# Set random seeds for reproducibility
RANDOM_SEED: int = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 2. Model loading
Using the GAIA model (Gemma-3-Gaia-PT-BR-4b), a Brazilian Portuguese Multimodal LLM, via Ollama, which runs on a remote server.

In [31]:
class OllamaClient:
    """
    Client wrapper for Ollama server.
    """

    def __init__(self, server_url=None, model_name="brunoconterato/Gemma-3-Gaia-PT-BR-4b-it:f16"):
        """
        Initialize Ollama client.
        
        Args:
            server_url: Desktop IP (e.g., "http://192.168.1.100:11434")
                       If None, uses OLLAMA_SERVER env variable or localhost
            model_name: Ollama model to use
        """
        # Get server URL from env or parameter
        self.server_url = (
            server_url 
            or os.getenv("OLLAMA_SERVER", "http://localhost:11434")
        )
        if not self.server_url.startswith("http"):
            self.server_url = f"http://{self.server_url}"


        self.model_name = model_name
        self.api_url = f"{self.server_url}/api/generate"

    def generate(self, prompt, max_tokens=1024, temperature=0.7, stream=False):
        """
        Generate text using Ollama server.
        
        Args:
            prompt: Input text prompt
            max_tokens: Maximum tokens to generate
            temperature: Sampling temperature (0.0-1.0)
            stream: Whether to stream response
        
        Returns:
            Generated text string
        """
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": stream,
            "options": {
                "num_predict": max_tokens,
                "temperature": temperature,
                "top_p": 0.9
            }
        }
        
        try:
            response = requests.post(
                self.api_url, 
                json=payload,
                timeout=120  # 2 minute timeout
            )
            
            if response.status_code == 200:
                return response.json()["response"]
            else:
                raise Exception(
                    f"Generation failed: {response.status_code} - {response.text}"
                )

        except requests.exceptions.Timeout:
            print("Generation timed out. Try reducing max_tokens.")
            raise
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            raise

def load_gaia_model():
    """
    Initialize the GAIA model client via Ollama.
    
    Returns:
        OllamaClient: Client ready for inference
    """
    print("Initializing GAIA model client...")
    
    # The client will use OLLAMA_SERVER env variable if set
    # Otherwise defaults to localhost
    client = OllamaClient()
    
    # Test connection
    try:
        test_response = client.generate("Test", max_tokens=10, temperature=0.1)
        print(f"✓ Connection successful!")
        print(f"✓ Model: {client.model_name}")
        return client
    except Exception as e:
        print(f"✗ Connection failed: {e}")
        print(f"  Make sure Ollama is running at {client.server_url}")
        print(f"  Set OLLAMA_SERVER environment variable if using remote server")
        raise

# Load the model
model_client = load_gaia_model()

## 3. Load Seed Data
The data was manually taken from Sistema B database. Link: https://www.bcorporation.net/en-us/find-a-b-corp/?refinement%5BhqCountry%5D%5B0%5D=Brazil

In [32]:
file_path = 'seed_data/companies_mvv.json'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        # Load the JSON data from the file into a Python dictionary
        seed_companies = json.load(f)
    
    print("Successfully loaded data")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

print(seed_companies)

Successfully loaded data
[{'company': 'Ecovalor Consultoria e Assessoria em Sustentabiliade LTDA', 'sector': 'ESG Consulting', 'mission': 'Guiar organizações rumo às melhores práticas de sustentabilidade, atuando em todo Brasil e oferecendo suporte a empresas de diversos setores e tamanhos.', 'vision': None, 'values': ['Conectar', 'Solucionar', 'Inovar', 'Re(voluir)']}, {'company': 'Florestal Alto Uruguai', 'sector': 'ESG Consulting', 'mission': 'Entregar sustentabilidade com acurácia, inovação e simplicidade.', 'vision': 'Ser referência nos setores em que atuamos, admirada por nossos clientes e lembrada por fazer diferente.', 'values': ['Confiança', 'Transparência', 'Encantamento']}, {'company': 'Weber Ambiental', 'sector': 'ESG Consulting', 'mission': 'Contribuir para a melhoria da condição ambiental e social do país, incluindo áreas de vulnerabilidade, proporcionando saúde e qualidade de vida.', 'vision': 'Até 2030, ser reconhecida nacionalmente pela atuação e, principalmente, inova

## 4. Generate Unified Company Profile

To ensure cross-document coherence (Survey paper: sample-wise decomposition strategy)

In [ ]:
# =============================================================================
# CONFIGURATION: Company Parameters
# =============================================================================

# Set the fiscal year
FISCAL_YEAR: int = 2024

TARGET_PORTE = "PEQUENA"  # Change this to generate different company sizes

# Company sector/specialization
TARGET_SECTOR = "Consultoria ESG"  # Examples:
#"Consultoria ESG"
#"Consultoria em Estratégia Empresarial"
#"Consultoria em TI e Transformação Digital"
#"Consultoria em Recursos Humanos"
#"Consultoria Financeira e Contábil"
#"Consultoria em Marketing Digital"

# Validate selection
VALID_PORTES = ['MICROEMPRESA', 'PEQUENA', 'MEDIA', 'GRANDE']

if TARGET_PORTE not in VALID_PORTES:
    raise ValueError(f"Invalid TARGET_PORTE. Must be one of: {VALID_PORTES}")

print(f"  Company size: {TARGET_PORTE}")
print(f"  Sector: {TARGET_SECTOR}")
print(f"  Fiscal year: {FISCAL_YEAR}")

# BNDES/IBGE ranges
constraints = {
    'MICROEMPRESA': {
        'employees': (1, 9),
        'revenue': (50_000, 360_000),
    },
    'PEQUENA': {
        'employees': (10, 49),
        'revenue': (360_000, 4_800_000),
    },
    'MEDIA': {
        'employees': (50, 99),
        'revenue': (4_800_000, 300_000_000),
    },
    'GRANDE': {
        'employees': (100, 1000), # 1000 is arbitrary, IBGE classifies as 100+
        'revenue': (300_000_000, 10_000_000_000), # 10.000.000.000 is arbitrary, BNDES classifies as 300.000.000+
    }
}

# Auto-generate employee count and revenue within porte range
emp_min, emp_max = constraints[TARGET_PORTE]['employees']
rev_min, rev_max = constraints[TARGET_PORTE]['revenue']

TARGET_EMPLOYEES = random.randint(emp_min, emp_max)

# Use log-normal for revenue (more realistic distribution)
log_mean = np.log((rev_min + rev_max) / 2)
TARGET_REVENUE = np.clip(
    np.random.lognormal(log_mean, 0.3),
    rev_min,
    rev_max
)

print(f"  Employees: {TARGET_EMPLOYEES} (auto-generated within {emp_min}-{emp_max})")
print(f"  Revenue: R$ {TARGET_REVENUE:,.2f} (auto-generated within R$ {rev_min:,.0f}-{rev_max:,.0f})")

  Company size: PEQUENA
  Sector: Consultoria ESG
  Fiscal year: 2024
  Employees: 17 (auto-generated within 10-49)
  Revenue: R$ 2,994,578.97 (auto-generated within R$ 360,000-4,800,000)


In [34]:
def create_company_profile_prompt(seed_companies: List[Dict],
                                target_porte: str,
                                target_sector: str,
                                target_employees: int,
                                target_revenue: float) -> str:
    """
    Create prompt to generate a unified company profile.
    
    Args:
        seed_companies: Seed company data
        target_porte: Target BNDES classification
        target_sector: Target business sector/specialization
        target_employees: Generated employee count
        target_revenue: Generated revenue (BRL)

    Returns:
        Formatted prompt string
    """
    
    task_spec = """Você é um especialista em criar perfis organizacionais para empresas 
brasileiras. Sua tarefa é gerar um perfil sintético coerente e realista para uma empresa 
de **{target_sector}** com as características fornecidas."""
    
    # Fixed parameters
    fixed_params = f"""\n\n=== PARÂMETROS OBRIGATÓRIOS (USE EXATAMENTE) ===

**PORTE:** {target_porte}
**SETOR:** {target_sector}
**NÚMERO DE FUNCIONÁRIOS:** {target_employees}
**FATURAMENTO ANUAL:** R$ {target_revenue:,.2f}

IMPORTANTE: Use EXATAMENTE estes valores. NÃO invente números diferentes.
"""

    # Seed context
    seed_section = f"""\n\n=== EMPRESAS DE REFERÊNCIA (Exemplos do Setor no Brasil) ===
    
    As empresas abaixo são exemplos reais do setor de {target_sector} brasileiro.
    Use-as como REFERÊNCIA para estrutura e estilo. Gere conteúdo específico 
    para o setor: **{target_sector}**
    """
    
    for i, company in enumerate(seed_companies, 1):
        seed_section += f"\n{i}. {company['company']}\n"
        seed_section += f"   Missão: {company['mission'][:100]}...\n"
        if company['vision']:
            seed_section += f"   Visão: {company['vision'][:100]}...\n"
    
    # Instructions
    instructions = f"""\n\n=== TAREFA ===

Gere um perfil completo para UMA empresa sintética brasileira especializada em 
**{target_sector}** e classificada como **{target_porte}**.
Este perfil deve ser internamente consistente e realista.

INICIE SEMPRE COM:

**PORTE:** {target_porte}
**SETOR:** {target_sector}

DEPOIS gere o perfil CONSISTENTE com porte e setor:

**Nome da Empresa:** [Nome criativo e realista, relacionado a {target_sector}]

**Localização:** [Cidade e Estado brasileiro, foco no Sul do Brasil]

**Ano de Fundação:** [Entre 2015-2020]

**Número de Funcionários:** {target_employees}

**Faturamento Anual:** R$ {target_revenue:,.2f}

**Área de Atuação Geográfica:** [Regional/Estadual/Nacional - coerente com porte]

**Especializações:** [3-5 áreas específicas de {target_sector}]
Exemplo para ESG: carbono, certificações, treinamentos
Exemplo para TI: cloud, cybersecurity, integração de sistemas
Exemplo para Estratégia: reestruturação, M&A, transformação digital

**Segmentos de Clientes:** [3-4 tipos principais que contratam {target_sector}]

**Diferenciais Competitivos:** [2-3 pontos fortes únicos]

**Estrutura da Equipe:**
- Perfil técnico relevante para {target_sector}
- Principais expertises

**Parcerias Estratégicas:** [2-3 parcerias típicas do setor]

**Métricas de Impacto (últimos 12 meses):**
- Clientes atendidos: [número realista para {target_porte}]
- Projetos executados: [número realista]
- Horas de consultoria/treinamento: [número realista]
- Métrica relevante ao setor: [ex: CO2e evitado (ESG), sistemas implementados (TI), etc.]

REQUISITOS CRÍTICOS:
- Conteúdo deve ser ESPECÍFICO para {target_sector}
- Valores dentro das faixas BNDES para {target_porte}
- Use terminologia adequada ao setor escolhido
- Mantenha contexto brasileiro

FORMATO: Liste cada campo claramente. Seja específico e realista.
"""
    
    return task_spec + fixed_params + seed_section + instructions


def generate_company_profile(model_client: OllamaClient, seed_companies: List[Dict],
                            target_porte: str, target_sector: str,
                            target_employees: int, target_revenue: float) -> Dict[str, Any]:
    """
    Generate a unified company profile for consistent document generation.
    
    Args:
        model_client: OllamaClient instance
        seed_companies: Seed company data
        target_porte: Target BNDES classification
    
    Returns:
        Dictionary containing company profile data
    """
    print("GENERATING UNIFIED COMPANY PROFILE")
    
    prompt = create_company_profile_prompt(seed_companies,
                                           target_porte,
                                           target_sector,
                                           target_employees,
                                           target_revenue)
    
    profile_text = model_client.generate(
        prompt=prompt,
        max_tokens=1024,
        temperature=0.7
    )
    
    print("Company profile generated")
    print(profile_text)
    
    return {
        "profile_text": profile_text,
        "target_porte": target_porte,
        "target_sector": target_sector,
        "target_employees": target_employees,
        "target_revenue": target_revenue,
        "generation_timestamp": datetime.now().isoformat()
    }


# Generate the company profile
company_profile = generate_company_profile(
    model_client, 
    seed_companies,
    target_porte=TARGET_PORTE,
    target_sector=TARGET_SECTOR,
    target_employees=TARGET_EMPLOYEES,
    target_revenue=TARGET_REVENUE
)

GENERATING UNIFIED COMPANY PROFILE
Company profile generated
**PORTE:** PEQUENA
**SETOR:** Consultoria ESG

**Nome da Empresa:** Sustenta Brasil Consultoria

**Localização:** Curitiba, Paraná

**Ano de Fundação:** 2018

**Número de Funcionários:** 17

**Faturamento Anual:** R$ 2,994,578.97

**Área de Atuação Geográfica:** Sul do Brasil (com potencial de expansão para outras regiões)

**Especializações:**
1.  **Gestão de Carbono:** Cálculo, monitoramento e compensação de emissões de gases de efeito estufa.
2.  **Certificações ESG:** Preparação e obtenção de certificações como GRI, SASB e CDP.
3.  **Treinamentos e Workshops:** Desenvolvimento de programas de educação em sustentabilidade para empresas.
4.  **Avaliação de Sustentabilidade:** Análise do desempenho ambiental, social e de governança das empresas.

**Segmentos de Clientes:**
1.  Pequenas e médias empresas (PMEs) do setor agroindustrial.
2.  Empresas de logística e transporte.
3.  Startups com foco em impacto social e ambiental

## 5. Document 1: Mission, Vision & Values Generator

### 5.1 MVV Prompt

In [35]:
def create_mvv_prompt(seed_companies: List[Dict], company_profile: str, target_sector: str) -> str:
    """
    Create seed-guided prompt for Mission, Vision & Values generation.
    
    Args:
        seed_companies: List of seed company data
        company_profile: Generated company profile
        target_sector: Target sector setted

    Returns:
        Formatted prompt string
    """
    
    # Task specification
    task_spec = """Você é um especialista em estratégia organizacional e branding corporativo 
para empresas brasileiras. Sua tarefa é criar uma declaração de Missão, Visão e Valores para 
uma empresa de **{target_sector}** que seja CONSISTENTE com o perfil fornecido."""
    
    # Company profile section
    profile_section = f"""\n\n=== PERFIL DA EMPRESA (BASE OBRIGATÓRIA) ===

{company_profile}

ATENÇÃO: Sua declaração de Missão, Visão e Valores DEVE ser consistente com TODOS os 
elementos deste perfil (localização, porte, especializações, clientes, etc.)
"""
    
    # Seed companies (in-context examples)
    seed_section = "\n\n=== EMPRESAS DE REFERÊNCIA (Exemplos do Setor ESG no Brasil) ===\n"
    
    for i, company in enumerate(seed_companies, 1):
        seed_section += f"\n--- Empresa {i}: {company['company']} ---\n"
        seed_section += f"Missão: {company['mission']}\n"
        if company['vision']:
            seed_section += f"Visão: {company['vision']}\n"
        seed_section += f"Valores: {', '.join(company['values'])}\n"
    
    # Generation instructions
    instructions = """\n\n=== TAREFA ===

Com base no perfil da empresa e nos exemplos acima, gere:

**Missão:** [1-2 frases que reflitam o propósito DA EMPRESA DO PERFIL]
- Deve mencionar especializações do perfil
- Deve ser coerente com segmentos de clientes do perfil

**Visão:** [1-2 frases aspiracionais coerentes com o perfil]
- Deve refletir área de atuação geográfica do perfil
- Deve ser ambiciosa mas realista para o porte da empresa

**Valores:** [4-6 valores]
- Devem refletir os diferenciais competitivos do perfil
- Devem ser consistentes com a cultura organizacional implícita no perfil

REQUISITOS CRÍTICOS:
- COERÊNCIA TOTAL com o perfil fornecido
- NÃO copie diretamente os exemplos - gere conteúdo NOVO mas consistente
- Mantenha a distribuição de número de valores (3-5 valores, conforme exemplos)
- Use português brasileiro formal apropriado para documentos corporativos
- Preserve temas comuns observados (sustentabilidade, impacto, transparência, inovação)

FORMATO:
**Missão:**
[texto]

**Visão:**
[texto]

**Valores:**
- [valor 1]
- [valor 2]
- [valor 3]
- [valor 4]
[...]

-----
CRITICAL: Seja direto e sem preâmbulos.
"""
    
    return task_spec + profile_section + seed_section + instructions


# Test prompt creation
mvv_prompt = create_mvv_prompt(seed_companies, company_profile['profile_text'], target_sector=TARGET_SECTOR)
print("✓ Mission/Vision/Values prompt created")
print(f"  Prompt length: {len(mvv_prompt)} characters")

✓ Mission/Vision/Values prompt created
  Prompt length: 6766 characters


### 5.2 Generate MVV

In [36]:
def generate_mvv(model_client: OllamaClient, prompt: str) -> str:
    """
    Generate synthetic Mission, Vision & Values document.
    
    Args:
        model_client: OllamaClient instance
        prompt: Seed-guided generation prompt
    
    Returns:
        Generated MVV text
    """
    
    print("GENERATING DOCUMENT 1: Mission, Vision & Values")
    
    generated = model_client.generate(
        prompt=prompt,
        max_tokens=1024,  # Sufficient for MVV based on seed patterns
        temperature=0.7  # Balance between diversity and seed alignment
    )
    
    print("✓ Generation complete")
    return generated


# Generate the document
mvv_document = generate_mvv(model_client, mvv_prompt)
print(mvv_document)

GENERATING DOCUMENT 1: Mission, Vision & Values
✓ Generation complete
**Missão:**
A Sustenta Brasil Consultoria impulsiona empresas do Sul do Brasil a adotarem práticas de sustentabilidade robustas, com foco na gestão de carbono, certificações ESG e programas de educação. Guiamos as PMEs, empresas de logística, startups e ONGs na construção de estratégias de impacto ambiental e social, garantindo a transparência e a responsabilidade em cada etapa.

**Visão:**
Ser reconhecida como a principal consultoria em sustentabilidade do Sul do Brasil, admirada por sua abordagem personalizada e compromisso com a transformação das empresas em agentes de impacto positivo. Buscamos expandir nossa atuação, consolidando a Sustenta Brasil Consultoria como referência em inovação e resultados.

**Valores:**
- Impacto
- Transparência
- Inovação
- Confiança
- Responsabilidade



## 6. Document 2: DRE

**Pure Python Rule-Based Approach**
- Following: BNDES (2025), Lei Complementar 123/2006, IBGE (2025)
- Margins estimated from: IBISWorld (2024), Deltek (2023), Javalgi et al. (2015)

### 6.1 Parameters class

In [ ]:
# =============================================================================
# DRE Generation Note: Margin Validity
# =============================================================================
# 
# The financial margins used (52-65% gross, 8-28% operational) are based on
# professional services benchmarks, primarily researched for ESG consulting.
# These margins can serve as a baseline for all consulting sectors.
# 
# Sources: IBISWorld (2024), Deltek/SPI (2023), Javalgi et al. (2015)

class BNDESParameters:
    """Financial parameters based on BNDES classification and ESG consulting benchmarks."""
    
    classification = {
        'MICROEMPRESA': {
            'revenue_range': (50_000, 360_000),
            'employees': (1, 9),
            'gross_margin': (0.50, 0.58),
            'operational_margin': (0.06, 0.11),
            'net_margin': (0.03, 0.08),
        },
        'PEQUENA': {
            'revenue_range': (360_000, 4_800_000),
            'employees': (10, 49),
            'gross_margin': (0.52, 0.60),
            'operational_margin': (0.08, 0.13),
            'net_margin': (0.05, 0.10),
        },
        'MEDIA': {
            'revenue_range': (4_800_000, 300_000_000),
            'employees': (50, 99),
            'gross_margin': (0.56, 0.63),
            'operational_margin': (0.15, 0.20),
            'net_margin': (0.12, 0.17),
        },
        'GRANDE': {
            'revenue_range': (300_000_000, 10_000_000_000),
            # 10.000.000.000 is arbitrary, BNDES classifies as 300.000.000+
            'employees': (100, 1000), # 1000 is arbitrary, IBGE classifies as 100+
            'gross_margin': (0.58, 0.65),
            'operational_margin': (0.22, 0.28),
            'net_margin': (0.17, 0.23),
        }
    }
    
    tax_rates = {
        'pis': 0.0065,
        'cofins': 0.03,
        'iss': 0.05,
    }

### 6.2 Generate DRE

In [38]:
class SyntheticDREGenerator:
    """Rule-based synthetic DRE generator for Brazilian ESG consulting firms."""
    
    def __init__(self, random_seed: int):
        self.rng = np.random.default_rng(random_seed)
        self.params = BNDESParameters()

    def generate(self, porte: str, empresa_id: str,
                 num_funcionarios: int,
                 target_revenue: float) -> Dict:
        """
        Generate a single synthetic DRE.

        Args:
            porte: BNDES classification
            empresa_id: Company identifier
            num_funcionarios: Employee count (from config)
            target_revenue: Target gross revenue (from config)
        
        Returns:
            Dict with DRE line items and metrics
        """

        config = self.params.classification[porte]

        # 1. Revenue
        receita_bruta = target_revenue
        
        # 2. Tax deductions
        deducoes_receita = receita_bruta * (
            self.params.tax_rates['pis'] +
            self.params.tax_rates['cofins'] +
            self.params.tax_rates['iss'] +
            self.rng.uniform(0.01, 0.03) # Returns/cancellations
        )
        receita_liquida = receita_bruta - deducoes_receita
        
        # 3. Gross margin → Costs
        margem_bruta = self.rng.uniform(*config['gross_margin'])
        custos_servicos = receita_liquida * (1 - margem_bruta)
        lucro_bruto = receita_liquida - custos_servicos
        
        # 4. Operational margin → Operating expenses
        margem_operacional = self.rng.uniform(*config['operational_margin'])
        ebit = receita_liquida * margem_operacional
        despesas_operacionais = lucro_bruto - ebit
        
        # 5. Financial result
        resultado_financeiro = receita_liquida * self.rng.uniform(-0.015, 0.005)
        
        # 6. LAIR
        lair = ebit + resultado_financeiro
        
        # 7. Taxes
        if lair > 0:
            csll = lair * 0.09
            irpj = lair * 0.15
        else:
            csll = irpj = 0
        
        # 8. Net profit
        lucro_liquido = lair - csll - irpj
        margem_liquida = lucro_liquido / receita_liquida if receita_liquida > 0 else 0
                
        return {
            'empresa_id': empresa_id,
            'porte': porte,
            'ano': FISCAL_YEAR,
            'num_funcionarios': num_funcionarios,
            'receita_bruta': round(receita_bruta, 2),
            'deducoes_receita': round(deducoes_receita, 2),
            'receita_liquida': round(receita_liquida, 2),
            'custos_servicos': round(custos_servicos, 2),
            'lucro_bruto': round(lucro_bruto, 2),
            'despesas_operacionais': round(despesas_operacionais, 2),
            'ebit': round(ebit, 2),
            'resultado_financeiro': round(resultado_financeiro, 2),
            'lair': round(lair, 2),
            'csll': round(csll, 2),
            'irpj': round(irpj, 2),
            'lucro_liquido': round(lucro_liquido, 2),
            'margem_bruta': round(margem_bruta, 4),
            'margem_operacional': round(margem_operacional, 4),
            'margem_liquida': round(margem_liquida, 4),
        }
    
    def format_as_csv(self, dre: Dict) -> str:
        output = StringIO()
        writer = csv.writer(output)
        
        # Header
        writer.writerow(['Linha', 'Descrição', 'Valor (R$)'])
        
        # DRE Lines
        writer.writerow(['METADATA', 'Empresa ID', dre['empresa_id']])
        writer.writerow(['METADATA', 'Porte', dre['porte']])
        writer.writerow(['METADATA', 'Ano', dre['ano']])
        writer.writerow(['METADATA', 'Funcionários', dre['num_funcionarios']])
        writer.writerow(['', '', ''])  # Blank line
        
        writer.writerow([1, 'RECEITA BRUTA', dre['receita_bruta']])
        writer.writerow([2, '(-) Deduções da Receita', -dre['deducoes_receita']])
        writer.writerow([3, '= RECEITA LÍQUIDA', dre['receita_liquida']])
        writer.writerow([4, '(-) Custos dos Serviços Prestados', -dre['custos_servicos']])
        writer.writerow([5, '= LUCRO BRUTO', dre['lucro_bruto']])
        writer.writerow([6, '(-) Despesas Operacionais', -dre['despesas_operacionais']])
        writer.writerow([7, '= EBIT (Resultado Operacional)', dre['ebit']])
        writer.writerow([8, '(+/-) Resultado Financeiro', dre['resultado_financeiro']])
        writer.writerow([9, '= LAIR (Lucro Antes dos Impostos)', dre['lair']])
        writer.writerow([10, '(-) CSLL', -dre['csll']])
        writer.writerow([11, '(-) IRPJ', -dre['irpj']])
        writer.writerow([12, '= LUCRO LÍQUIDO', dre['lucro_liquido']])
        writer.writerow(['', '', ''])  # Blank line
        
        # Indicators
        writer.writerow(['INDICADOR', 'Margem Bruta', f"{dre['margem_bruta']*100:.2f}%"])
        writer.writerow(['INDICADOR', 'Margem Operacional (EBIT)', f"{dre['margem_operacional']*100:.2f}%"])
        writer.writerow(['INDICADOR', 'Margem Líquida', f"{dre['margem_liquida']*100:.2f}%"])
        
        return output.getvalue()

# Generate DRE
print("GENERATING DOCUMENT 2: DRE (Demonstração do Resultado do Exercício)")

# Initialize DRE generator
dre_generator = SyntheticDREGenerator(random_seed=RANDOM_SEED)

# Generate DRE
dre_data = dre_generator.generate(
    porte=TARGET_PORTE,
    empresa_id="SYNTH_ESG_001",
    num_funcionarios=TARGET_EMPLOYEES,
    target_revenue=TARGET_REVENUE,
)

# Format as markdown
dre_document = dre_generator.format_as_csv(dre_data)

print("✓ DRE generated")
print(f"  Actual Revenue: R$ {dre_data['receita_liquida']:,.2f}")
print(f"  EBIT Margin: {dre_data['margem_operacional']*100:.1f}%")
print(f"  Net Profit: R$ {dre_data['lucro_liquido']:,.2f}")

# Display preview
print("\n" + dre_document[:500] + "...")

GENERATING DOCUMENT 2: DRE (Demonstração do Resultado do Exercício)
✓ DRE generated
  Actual Revenue: R$ 2,659,248.65
  EBIT Margin: 12.3%
  Net Profit: R$ 246,317.47

Linha,Descrição,Valor (R$)
METADATA,Empresa ID,SYNTH_ESG_001
METADATA,Porte,PEQUENA
METADATA,Ano,2024
METADATA,Funcionários,17
,,
1,RECEITA BRUTA,2994578.97
2,(-) Deduções da Receita,-335330.32
3,= RECEITA LÍQUIDA,2659248.65
4,(-) Custos dos Serviços Prestados,-1183072.4
5,= LUCRO BRUTO,1476176.25
6,(-) Despesas Operacionais,-1149275.09
7,= EBIT (Resultado Operacional),326901.16
8,(+/-) Resultado Financeiro,-2799.23
9,= LAIR (Lucro Antes dos Impostos),324101.93
10,(-) CSLL,-29169....


## 7. Document 3: Social Impact Report Generator

### 7.1 Social Impact Report Prompt

In [ ]:
def create_impact_report_prompt(seed_companies: List[Dict], 
                                company_profile: str,
                                mvv_document: str,
                                dre_data: Dict) -> str:
    """
    Create seed-guided prompt for Social Impact Report generation.
        
    Args:
        seed_companies: Seed company data
        company_profile: Generated company profile,
        mvv_document: Generated MVV,
        dre_data: Generated DRE

    Returns:
        Formatted prompt string
    """
    
    # Task specification
    task_spec = """Você é um especialista em relatórios de impacto ESG. Sua tarefa é criar 
um Relatório de Impacto Social que seja TOTALMENTE CONSISTENTE com o perfil organizacional 
e a missão/visão/valores já definidos."""
    
    # Profile and MVV context
    context_section = f"""\n\n=== PERFIL DA EMPRESA (BASE OBRIGATÓRIA) ===

{company_profile}

=== MISSÃO, VISÃO E VALORES DA EMPRESA ===

{mvv_document}

=== DESEMPENHO FINANCEIRO ({FISCAL_YEAR}) ===

- Receita Líquida: R$ {dre_data['receita_liquida']:,.2f}
- Número de Funcionários: {dre_data['num_funcionarios']}
- Margem Operacional: {dre_data['margem_operacional']*100:.1f}%
- Lucro Líquido: R$ {dre_data['lucro_liquido']:,.2f}

ATENÇÃO: O relatório de impacto DEVE:
- Usar as métricas exatas do perfil
- Refletir as especializações mencionadas
- Demonstrar progresso alinhado com a missão
- Citar parcerias mencionadas no perfil
- Mencionar o faturamento (ordem de grandeza correta)
- Refletir o porte da empresa (funcionários)
- Demonstrar impacto proporcional à receita
"""
    
    # Seed themes
    seed_section = "\n\n=== TEMAS DE IMPACTO (Referência do Setor) ===\n"
    for i, company in enumerate(seed_companies[:3], 1):
        seed_section += f"{i}. {company['mission'][:100]}...\n"
    
    # Instructions
    instructions = f"""\n\n=== TAREFA ===

Gere um Relatório de Impacto Social para a empresa do perfil acima.
O relatório DEVE seguir a estrutura:

========================COMEÇO DA ESTRUTURA DO RELATÓRIO========================

# RELATÓRIO DE IMPACTO SOCIAL - {FISCAL_YEAR}
**[Nome da Empresa]**
---

## 1. MENSAGEM DA LIDERANÇA

[2-3 parágrafos]
- Reflita a missão e visão da empresa
- Destaque principais conquistas do ano
- Mencione desafios enfrentados
- Tom consistente com valores organizacionais

---

## 2. SOBRE A ORGANIZAÇÃO

### 2.1 Perfil Institucional
- **Fundação:** [Ano - do perfil]
- **Sede:** [Localização - do perfil]
- **Área de Atuação:** [Geográfica - do perfil]
- **Porte:** [Faturamento e funcionários - do DRE]
- **Especializações:** [Listar - do perfil]

### 2.2 Governança
- **Estrutura de Gestão:** [Simples: diretoria executiva, conselho consultivo]
- **Principais Políticas:** [2-3 políticas alinhadas com valores: ética, transparência, etc.]

---

## 3. INDICADORES DE IMPACTO CONSOLIDADOS

### 3.1 Resumo Executivo

| Indicador | Meta | Realizado | Status |
|-----------|------|-----------|--------|
| Clientes Atendidos | [do perfil] | [do perfil] | ✓ |
| Projetos Executados | [do perfil] | [do perfil] | ✓ |
| Horas de Capacitação | [do perfil] | [do perfil] | ✓ |
| CO2eq Evitado (ton) | [do perfil] | [do perfil] | ✓ |
| Pessoas Capacitadas | [derivar: horas/8] | [valor] | ✓ |

### 3.2 Detalhamento por Eixo ESG

**EIXO AMBIENTAL (E)**
- **Mudanças Climáticas:** CO2eq evitado através de projetos de clientes: [número do perfil] toneladas
- **Metodologia:** Estimativas baseadas em GHG Protocol Escopo 3
- **Principais Projetos:** [2 exemplos alinhados com especializações]

**EIXO SOCIAL (S)**
- **Beneficiários Diretos:** [clientes do perfil] empresas e [pessoas capacitadas] profissionais
- **Desenvolvimento de Competências:** [horas do perfil] horas de treinamento em ESG
- **Alcance Geográfico:** [área de atuação do perfil]

**EIXO GOVERNANÇA (G)**
- **Transparência:** Relatórios anuais publicados
- **Ética:** Código de conduta implementado
- **Certificações:** [Mencionar se houver no perfil, senão: "em processo"]

---

## 4. ALINHAMENTO COM OS ODS

### 4.1 Contribuição aos Objetivos de Desenvolvimento Sustentável

**ODS Primários (Impacto Direto):**

**ODS 13 - Ação contra a Mudança Global do Clima**
- Contribuição: Apoio a empresas na redução de emissões de GEE
- Indicador: [CO2eq evitado do perfil] toneladas
- Meta 13.3: Educação e conscientização climática

**ODS 12 - Consumo e Produção Responsáveis**
- Contribuição: Consultoria em economia circular e gestão sustentável
- Indicador: [projetos executados] projetos de eficiência
- Meta 12.6: Incentivo à adoção de práticas sustentáveis

**ODS 8 - Trabalho Decente e Crescimento Econômico**
- Contribuição: Capacitação profissional em sustentabilidade
- Indicador: [pessoas capacitadas] profissionais formados
- Meta 8.2: Produtividade econômica através da inovação

**ODS Secundários (Impacto Indireto):**
- ODS 17: Parcerias para implementação (colaboração com [parcerias do perfil])
- ODS 4: Educação de qualidade (programas de capacitação)

---

## 5. DESTAQUES QUALITATIVOS

### Projeto 1: [Nome relacionado às especializações]
- **Cliente:** [Segmento do perfil - ex: indústria manufatureira]
- **Escopo:** [Especialização específica - ex: inventário de GEE]
- **Resultado:** [Impacto mensurável]

### Projeto 2: [Nome relacionado às especializações]
- **Cliente:** [Outro segmento do perfil]
- **Escopo:** [Outra especialização]
- **Resultado:** [Impacto mensurável]

### Projeto 3: [Nome relacionado às especializações]
- **Cliente:** [Segmento do perfil]
- **Escopo:** [Especialização]
- **Resultado:** [Impacto mensurável]

---

## 6. ENGAJAMENTO DE STAKEHOLDERS

### 6.1 Parcerias Estratégicas
[Mencionar EXATAMENTE as parcerias listadas no perfil]

---

## 7. DESAFIOS E APRENDIZADOS

### 7.1 Principais Desafios
- [Desafio 1 realista para empresa do porte: ex: recursos limitados]
- [Desafio 2 do setor: ex: conscientização de PMEs]

### 7.2 Aprendizados
- [Aprendizado relevante para ESG]
- [Melhoria de processo ou metodologia]

---

## 8. COMPROMISSOS FUTUROS

### 8.1 Metas para [Ano+1]

**Ambiental:**
- Aumentar CO2eq evitado em [10-20%]
- [Meta específica alinhada com visão]

**Social:**
- Capacitar [+20-30%] profissionais
- Expandir para [nova região se aplicável]

**Governança:**
- [Meta de certificação ou política]
- [Melhoria de transparência/reporte]

### 8.2 Visão de Longo Prazo
[1 parágrafo conectando metas com a VISÃO da empresa]

---

**Metodologia de Medição:**
Este relatório segue diretrizes da ABNT PR 2030 para reporte ESG, com indicadores 
alinhados aos Objetivos de Desenvolvimento Sustentável (ODS) da ONU. Dados verificados 
internamente e validados através de pesquisas de satisfação com clientes.

========================FIM DA ESTRUTURA DO RELATÓRIO========================

REQUISITOS CRÍTICOS:

1. **CONSISTÊNCIA TOTAL:**
   - Use números EXATOS do perfil (clientes, projetos, horas, CO2)
   - Faturamento e funcionários do DRE (se disponível)
   - Especializações do perfil nos destaques de projetos
   - Parcerias exatas listadas no perfil

2. **PROPORCIONALIDADE:**
   - Impacto deve ser REALISTA para o porte da empresa
   - Não exagere números ou alcance
   - Empresa pequena = impacto pequeno mas focado

3. **PADRÕES BRASILEIROS:**
   - Mencione ABNT PR 2030
   - Alinhamento com ODS (obrigatório para ESG no Brasil)
   - Use terminologia brasileira (ESG, GEE, ODS, etc.)

4. **FORMATO:**
   - Markdown estruturado
   - Tabelas para indicadores
   - Seções numeradas claramente

5. **TOM:**
   - Profissional e transparente
   - Reconheça limitações/desafios
   - Coerente com missão e valores

CRITICAL: NÃO inclua preâmbulos, explicações ou texto introdutório antes do título. 
Comece a geração DIRETAMENTE com:

# RELATÓRIO DE IMPACTO SOCIAL [ANO]
[...]
"""
    
    return task_spec + context_section + seed_section + instructions


# Create prompt
impact_report_prompt = create_impact_report_prompt(
    seed_companies, 
    company_profile['profile_text'],
    mvv_document,
    dre_data
)
print("Social Impact Report prompt created")
print(f"Prompt length: {len(impact_report_prompt)} characters")

NameError: name 'List' is not defined

### 7.2 Generate Social Impact Report

In [40]:
def generate_impact_report(model_client: OllamaClient, prompt: str) -> str:
    """
    Generate synthetic Social Impact Report.
    
    Args:
        model_client: OllamaClient instance
        prompt: Seed-guided generation prompt
    
    Returns:
        Generated impact report text
    """
    
    print("GENERATING DOCUMENT 3: Social Impact Report")
    
    generated = model_client.generate(
        prompt=prompt,
        max_tokens=2048,
        temperature=0.7
    )
    
    print("✓ Generation complete")
    return generated


# Generate the document
impact_report_document = generate_impact_report(model_client, impact_report_prompt)
print(impact_report_document)

GENERATING DOCUMENT 3: Social Impact Report
✓ Generation complete
# Relatório de Impacto Social - Sustenta Brasil Consultoria

**1. Mensagem da Liderança**

A Sustenta Brasil Consultoria foi fundada em Curitiba, Paraná, em 2018, com a missão de impulsionar empresas do Sul do Brasil a adotarem práticas de sustentabilidade robustas. Nosso trabalho se fundamenta na gestão de carbono, certificações ESG e programas de educação, visando a transformação das PMEs, empresas de logística, startups e ONGs em agentes de impacto positivo. Alinhados com a nossa visão de sermos a principal consultoria em sustentabilidade da região, nos comprometemos com a transparência, a responsabilidade e a inovação, guiando nossos clientes rumo a um futuro mais sustentável.

Nosso compromisso com a ética e a responsabilidade social nos impulsiona a buscar a excelência em cada projeto, garantindo que a Sustenta Brasil Consultoria contribua significativamente para a melhoria da condição ambiental e social do país, e

## 8. Document 4: Business Model Canvas Generator

### 8.1 Business Model Canvas Prompt

In [ ]:
def create_bmc_prompt(seed_companies: List[Dict],
                        company_profile: str,
                        mvv_document: str,
                        dre_data: Dict) -> str:
    """
    Create seed-guided prompt for Business Model Canvas generation.
    Uses Osterwalder framework (standard) + seed data for terminology.
    
    Args:
        seed_companies: Seed company data
    
    Returns:
        Formatted prompt string
    """
    
    # Task specification
    task_spec = """Você é um especialista em Business Model Canvas para empresas ESG brasileiras. 
Sua tarefa é criar um BMC TOTALMENTE CONSISTENTE com o perfil, missão e modelo de negócio 
da empresa."""
    
    # Context
    context_section = f"""

=== PERFIL DA EMPRESA (BASE OBRIGATÓRIA) ===

{company_profile}

=== MISSÃO E VISÃO DA EMPRESA ===

{mvv_document}

=== DESEMPENHO FINANCEIRO ({FISCAL_YEAR}) ===

- Receita Líquida: R$ {dre_data['receita_liquida']:,.2f}
- Funcionários: {dre_data['num_funcionarios']}
- Margem Operacional: {dre_data['margem_operacional']*100:.1f}%

ATENÇÃO: O BMC DEVE refletir EXATAMENTE:
- Fontes de Receita → somam aproximadamente R$ {dre_data['receita_liquida']:,.0f}
- Estrutura de Custos → coerente com margem de {dre_data['margem_operacional']*100:.0f}%
- Recursos Principais → equipe de {dre_data['num_funcionarios']} pessoas
- Especializações → Proposta de Valor
- Segmentos de clientes mencionados → Segmentos de Clientes
- Parcerias listadas → Parcerias Principais
- Porte e faturamento → Estrutura de Custos e Fontes de Receita
- Área geográfica → Canais e Relacionamento
"""
    
    # Seed context
    seed_section = "\n\n=== EXEMPLOS DO SETOR (Referência) ===\n"
    for i, company in enumerate(seed_companies[:3], 1):
        seed_section += f"{i}. {company['company']}: {company['mission'][:80]}...\n"
    
    # Instructions
    instructions = """

=== TAREFA ===

Gere um Business Model Canvas para a empresa do perfil acima.

Para cada bloco (3-5 itens):

**1. SEGMENTOS DE CLIENTES**
- Use EXATAMENTE os segmentos mencionados no perfil

**2. PROPOSTA DE VALOR**
- Baseie-se nas especializações do perfil
- Reflita a missão da empresa

**3. CANAIS**
- Coerentes com área geográfica do perfil

**4. RELACIONAMENTO COM CLIENTES**
- Apropriado para o porte da empresa

**5. FONTES DE RECEITA**
- Alinhadas com especializações
- Realistas para o faturamento mencionado no perfil

**6. RECURSOS PRINCIPAIS**
- Inclua equipe técnica mencionada no perfil
- Inclua parcerias estratégicas do perfil

**7. ATIVIDADES-CHAVE**
- Derivadas das especializações

**8. PARCERIAS PRINCIPAIS**
- Use EXATAMENTE as parcerias listadas no perfil
- Adicione 1-2 coerentes com o setor

**9. ESTRUTURA DE CUSTOS**
- Realista para empresa do porte mencionado

REQUISITOS CRÍTICOS:
- CONSISTÊNCIA TOTAL com perfil, MVV e métricas
- Não invente informações que contradigam o perfil

FORMATO:
## BUSINESS MODEL CANVAS

### [Bloco]
- Item 1
- Item 2
[...]

-----
CRITICAL: Seja direto e sem preâmbulos. 
"""
    
    return task_spec + context_section + seed_section + instructions


# Create prompt
bmc_prompt = create_bmc_prompt(seed_companies,
    company_profile['profile_text'],
    mvv_document,
    dre_data
)
print("✓ Business Model Canvas prompt created")
print(f"  Prompt length: {len(bmc_prompt)} characters")

✓ Business Model Canvas prompt created
  Prompt length: 5463 characters


### 8.2 Generate Business Model Canvas

In [42]:
def generate_bmc(model_client: OllamaClient, prompt: str) -> str:
    """
    Generate synthetic Business Model Canvas.
    
    Args:
        model_client: OllamaClient instance
        prompt: Seed-guided generation prompt
    
    Returns:
        Generated BMC text
    """
    print("GENERATING DOCUMENT 4: Business Model Canvas")
    
    generated = model_client.generate(
        prompt=prompt,
        max_tokens=1536,
        temperature=0.7
    )
    
    print("✓ Generation complete")
    return generated


# Generate the document
bmc_document = generate_bmc(model_client, bmc_prompt)
print(bmc_document)

GENERATING DOCUMENT 4: Business Model Canvas
✓ Generation complete
## BUSINESS MODEL CANVAS
### 1. SEGMENTOS DE CLIENTES
- Pequenas e médias empresas (PMEs) do setor agroindustrial (Sul do Brasil)
- Empresas de logística e transporte (Sul do Brasil)
- Startups com foco em impacto social e ambiental (Sul do Brasil)
- Organizações não governamentais (ONGs) focadas em sustentabilidade (Sul do Brasil)

### 2. PROPOSTA DE VALOR
- Consultoria personalizada em gestão de carbono, certificações ESG e avaliação de sustentabilidade.
- Auxílio na obtenção de certificações GRI, SASB e CDP.
- Programas de educação e treinamento em sustentabilidade para empresas.
- Compromisso com a transparência, responsabilidade e impacto ambiental e social.
- Soluções inovadoras e adaptadas às necessidades específicas de cada cliente.

### 3. CANAIS
- Vendas diretas (equipe de consultores)
- Eventos e feiras do setor agroindustrial e de logística
- Parcerias com associações e câmaras de comércio
- Marketing digita

## 9. SWOT Analysis Generator

### 9.1 SWOT Prompt

In [43]:
def create_swot_prompt(seed_companies: List[Dict],
                        company_profile: str,
                        mvv_document: str,
                        bmc_document: str,
                        dre_data: Dict) -> str:
    """
    Create seed-guided prompt for SWOT Analysis generation.
    
    Args:
        seed_companies: List of real company examples
        company_profile: Generated company profile
        mvv_document: Generated MVV
        bmc_document: Generated BMC
    
    Returns:
        Formatted prompt string    
    """
    
    # Task specification
    task_spec = """Você é um especialista em análise estratégica para empresas ESG brasileiras. 
Sua tarefa é criar uma análise SWOT TOTALMENTE CONSISTENTE com o perfil, missão, visão, valores 
e modelo de negócio já definidos."""
    
    # Full context
    context_section = f"""

=== PERFIL DA EMPRESA ===

{company_profile}

=== MISSÃO, VISÃO E VALORES ===

{mvv_document}

=== DESEMPENHO FINANCEIRO ({FISCAL_YEAR})===

- Receita: R$ {dre_data['receita_liquida']:,.2f}
- Margem Operacional: {dre_data['margem_operacional']*100:.1f}%
- Lucro Líquido: R$ {dre_data['lucro_liquido']:,.2f}

=== MODELO DE NEGÓCIO (BMC) ===

{bmc_document[:800]}... [resumo]

ATENÇÃO: O SWOT DEVE:
- FORÇAS: Refletir diferenciais e recursos do perfil/BMC. Incluir desempenho financeiro se margem > 15%
- FRAQUEZAS: Considerar limitações do porte e área geográfica. Mencionar limitações financeiras se margem < 10%
- OPORTUNIDADES: Alinhadas com visão e especializações
- AMEAÇAS: Relevantes para o mercado e posicionamento da empresa
- Ser consistente com o tamanho da empresa
"""
    
    # Market context
    market_section = """\n\n=== CONTEXTO DE MERCADO ESG BRASILEIRO ===

- Competição crescente
- Demanda regulatória aumentando
- Entrada de grandes consultorias
- Oportunidades em carbono e certificações
"""
    
    # Instructions
    instructions = """\n\n=== TAREFA ===

Gere uma análise SWOT consistente com TODOS os documentos anteriores.

**FORÇAS** (5 itens)
- Derive dos "Diferenciais Competitivos" e "Recursos Principais" do BMC
- Mencione parcerias estratégicas do perfil

**FRAQUEZAS** (5 itens)
- Considere o porte da empresa
- Considere limitações geográficas mencionadas

**OPORTUNIDADES** (5 itens)
- Alinhadas com a VISÃO da empresa
- Relacionadas às especializações

**AMEAÇAS** (5 itens)
- Relevantes para o setor e posicionamento

REQUISITOS CRÍTICOS:
- CONSISTÊNCIA com perfil, MVV e BMC
- Não contradiga informações anteriores
- Seja específico e realista

FORMATO:
## ANÁLISE SWOT

### FORÇAS
- [item]
[...]

### FRAQUEZAS
- [item]
[...]

### OPORTUNIDADES
- [item]
[...]

### AMEAÇAS
- [item]
[...]

-----
CRITICAL: Seja direto e sem preâmbulos. 
"""
    
    return task_spec + context_section + market_section + instructions


# Create SWOT prompt
swot_prompt = create_swot_prompt(seed_companies,
    company_profile['profile_text'],
    mvv_document,
    bmc_document,
    dre_data
)
print("✓ SWOT Analysis prompt created")
print(f"  Prompt length: {len(swot_prompt)} characters")

✓ SWOT Analysis prompt created
  Prompt length: 5752 characters


### 9.2 Generate SWOT Analysis

In [44]:
def generate_swot(model_client: OllamaClient, prompt: str) -> str:
    """
    Generate synthetic SWOT Analysis document.
    
    Args:
        model_client: OllamaClient instance
        prompt: Seed-guided generation prompt
    
    Returns:
        Generated SWOT analysis text
    """
    print("GENERATING DOCUMENT 5: SWOT Analysis")
    
    generated = model_client.generate(
        prompt=prompt,
        max_tokens=1536,
        temperature=0.7
    )
    
    print("✓ Generation complete")
    return generated


# Generate the document
swot_document = generate_swot(model_client, swot_prompt)
print(swot_document)

GENERATING DOCUMENT 5: SWOT Analysis
✓ Generation complete
## ANÁLISE SWOT - Sustenta Brasil Consultoria

### FORÇAS
1. **Abordagem Personalizada:** A Sustenta Brasil Consultoria se destaca por oferecer soluções de sustentabilidade adaptadas às necessidades específicas de cada cliente, diferenciando-se de consultorias mais generalistas.
2. **Equipe Qualificada:** Uma equipe experiente em gestão de carbono, certificações ESG e avaliação de sustentabilidade garante alta qualidade dos serviços.
3. **Parcerias Estratégicas:** Colaborações com a Associação Brasileira de Carbono (ABC), o Instituto Ethos e a Universidade Federal do Paraná (UFPR) impulsionam a inovação e o acesso a conhecimento especializado.
4. **Compromisso com o Impacto:** A empresa demonstra um compromisso genuíno com a sustentabilidade, refletido em suas métricas de impacto e na busca por resultados mensuráveis.
5. **Reputação em Crescimento:** A empresa tem construído uma reputação positiva no mercado local, especialment

## 10. Save Generated Documents

In [45]:
def save_documents(mvv: str, dre: str, swot: str, impact: str, bmc: str, 
                       output_dir: str = "synthetic_documents"):
    """
    Save all generated synthetic documents.
    
    Args:
        mvv: Mission/Vision/Values text
        dre: DRE text
        swot: SWOT Analysis text
        impact: Social Impact Report text
        bmc: Business Model Canvas text
        output_dir: Output directory path
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Save Document 1: MVV
    with open(os.path.join(output_dir, "doc1_mission_vision_values.md"), 'w', encoding='utf-8') as f:
        f.write(f"# Missão, Visão e Valores\n\n")
        f.write(f"*Documento Sintético | Gerado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
        f.write(mvv)
    
    # Save Document 2: DRE (CSV format)
    with open(os.path.join(output_dir, "doc2_dre.csv"), 'w', encoding='utf-8') as f:
        f.write(dre_document)
    
    # Save Document 3: Impact Report
    with open(os.path.join(output_dir, "doc3_social_impact_report.md"), 'w', encoding='utf-8') as f:
        f.write(f"# Relatório de Impacto Social\n\n")
        f.write(f"*Documento Sintético | Gerado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
        f.write(impact)
    
    # Save Document 4: BMC
    with open(os.path.join(output_dir, "doc4_business_model_canvas.md"), 'w', encoding='utf-8') as f:
        f.write(f"# Business Model Canvas\n\n")
        f.write(f"*Documento Sintético | Gerado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
        f.write(bmc)
    
    # Save Document 5: SWOT
    with open(os.path.join(output_dir, "doc5_swot_analysis.md"), 'w', encoding='utf-8') as f:
        f.write(f"# Análise SWOT\n\n")
        f.write(f"*Documento Sintético | Gerado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
        f.write(swot)
    
    # Save metadata
    metadata = {
        "generation_date": datetime.now().isoformat(),
        "model": "brunoconterato/Gemma-3-Gaia-PT-BR-4b-it:f16",
        "methodology": "Hybrid: LLM for narrative docs, Rule-based for DRE",
        "seed_companies_count": len(seed_companies),
        "seed_companies_sector": TARGET_SECTOR,
        "documents_generated": [
            "Mission_Vision_Values",
            "DRE (Rule-based Python)",
            "Social_Impact_Report",
            "Business_Model_Canvas",
            "SWOT_Analysis"
        ],
        "configuration": {
            "target_porte": TARGET_PORTE,
            "target_sector": TARGET_SECTOR,
            "fiscal_year": FISCAL_YEAR,
            "random_seed": RANDOM_SEED
        },
        "generation_parameters": {
            "temperature": 0.7,
            "random_seed": 42
        },
        "references": [
            "DocGenie: A Framework for High-Fidelity Synthetic Document Generation (Harikrishnan et al., 2025)",
            "On LLMs-Driven Synthetic Data Generation, Curation, and Evaluation: A Survey (Long et al., 2024)",
            "SynEval: A Multi-Faceted Evaluation Framework (Paper 3)"
        ]
    }
    
    metadata_path = os.path.join(output_dir, "generation_metadata.json")
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"\n✓ Synthetic documents saved to: {output_dir}/")


# Save all documents
save_documents(mvv_document, dre_document, swot_document, impact_report_document, bmc_document)


✓ Synthetic documents saved to: synthetic_documents/


## 11. Validation

### 11.1 Cross-Document Consistency Validation

In [ ]:
def validate_dre_consistency(dre_data: Dict,
                            expected_porte: str,
                            expected_employees: int,
                            expected_revenue: float) -> Dict:
    """Validate DRE matches configuration."""
    
    print("\n" + "="*70)
    print("VALIDATION: DRE Configuration Consistency")
    print("="*70)
    
    results = {}
    
    # Check 1: Porte
    print("\n📋 Check 1: Porte")
    porte_match = (dre_data['porte'] == expected_porte)
    print(f"  {'✓ PASS' if porte_match else '✗ FAIL'}: {dre_data['porte']}")
    results['porte_match'] = porte_match
    
    # Check 2: Employees
    print("\n📋 Check 2: Employee Count")
    emp_match = (dre_data['num_funcionarios'] == expected_employees)
    print(f"  {'✓ PASS' if emp_match else '✗ FAIL'}: {dre_data['num_funcionarios']}")
    results['employees_match'] = emp_match
    
    # Check 3: Revenue matches exactly
    print("\n📋 Check 3: Revenue Match")
    revenue_match = np.isclose(dre_data['receita_bruta'], expected_revenue, rtol=0.001)
    print(f"  {'✓ PASS' if revenue_match else '✗ FAIL'}: R$ {dre_data['receita_bruta']:,.2f}")
    results['revenue_match'] = revenue_match
    
    # Check 4: Accounting identities
    print("\n📋 Check 4: BR GAAP Identities")
    
    checks = [
        ('Receita Líquida', dre_data['receita_liquida'], 
         dre_data['receita_bruta'] - dre_data['deducoes_receita']),
        ('Lucro Bruto', dre_data['lucro_bruto'], 
         dre_data['receita_liquida'] - dre_data['custos_servicos']),
        ('Lucro Líquido', dre_data['lucro_liquido'], 
         dre_data['lair'] - dre_data['csll'] - dre_data['irpj'])
    ]
    
    all_valid = True
    for name, actual, expected in checks:
        if not np.isclose(actual, expected, rtol=0.01):
            print(f"  ✗ FAIL: {name}")
            all_valid = False
    
    if all_valid:
        print("  ✓ PASS: All identities valid")
    
    results['accounting_valid'] = all_valid
    
    print("="*70)
    return results

# Run validation
dre_validation = validate_dre_consistency(
    dre_data,
    expected_porte=TARGET_PORTE,
    expected_employees=TARGET_EMPLOYEES,
    expected_revenue=TARGET_REVENUE
)


VALIDATION: DRE Configuration Consistency

📋 Check 1: Porte
  ✓ PASS: PEQUENA

📋 Check 2: Employee Count
  ✓ PASS: 17

📋 Check 3: Revenue Proximity
  ✓ PASS: R$ 2,994,578.97
    (Target: R$ 2,994,578.97, diff: 0.0%)

📋 Check 4: BR GAAP Identities
  ✓ PASS: All identities valid


In [47]:
def validate_cross_document_consistency(
    company_profile: str,
    mvv: str,
    impact: str,
    bmc: str,
    swot: str
) -> Dict[str, Any]:
    """
    Validate cross-document consistency.
    
    Args:
        company_profile: Generated company profile
        mvv: Mission/Vision/Values
        impact: Social Impact Report
        bmc: Business Model Canvas
        swot: SWOT Analysis
    
    Returns:
        Dictionary with consistency check results
    """
    
    print("\n" + "="*70)
    print("VALIDATION: Cross-Document Consistency")
    print("="*70)
    
    results = {
        "checks_passed": 0,
        "checks_total": 0,
        "details": {}
    }
    
    # Extract company name from profile (if present)
    profile_lower = company_profile.lower()
    
    # Check 1: Company size consistency
    print("\n📋 Check 1: Company Size Consistency")
    size_mentions = []
    if "pequena" in profile_lower or "pequeno" in profile_lower:
        size_mentions.append("profile: pequena")
    if "média" in profile_lower or "medio" in profile_lower:
        size_mentions.append("profile: média")
    
    # Check if SWOT mentions size appropriately
    swot_lower = swot.lower()
    swot_mentions_size = ("pequena" in swot_lower or "média" in swot_lower or 
                          "porte" in swot_lower or "recurso" in swot_lower)
    
    size_consistent = len(size_mentions) > 0
    results["checks_total"] += 1
    if size_consistent:
        results["checks_passed"] += 1
        print(f"  ✓ PASS: Size indicators found - {size_mentions}")
    else:
        print(f"  ⚠ WARNING: Size not clearly specified")
    results["details"]["size_consistency"] = size_consistent
    
    # Check 2: Geographic scope consistency
    print("\n📋 Check 2: Geographic Scope Consistency")
    geo_keywords = ["regional", "estadual", "nacional", "rio grande do sul", 
                    "sul", "brasil", "estado"]
    
    geo_in_profile = sum(1 for kw in geo_keywords if kw in profile_lower)
    geo_in_mvv = sum(1 for kw in geo_keywords if kw in mvv.lower())
    geo_in_bmc = sum(1 for kw in geo_keywords if kw in bmc.lower())
    
    geo_consistent = geo_in_profile > 0 and (geo_in_mvv > 0 or geo_in_bmc > 0)
    results["checks_total"] += 1
    if geo_consistent:
        results["checks_passed"] += 1
        print(f"  ✓ PASS: Geographic scope mentioned across documents")
    else:
        print(f"  ⚠ WARNING: Geographic scope may not be consistent")
    results["details"]["geography_consistency"] = geo_consistent
    
    # Check 3: Service/Specialization alignment
    print("\n📋 Check 3: Service/Specialization Alignment")
    service_keywords = ["consultoria", "treinamento", "capacitação", "carbono",
                       "certificação", "auditoria", "esg", "sustentabilidade"]
    
    services_in_profile = [kw for kw in service_keywords if kw in profile_lower]
    services_in_mvv = [kw for kw in service_keywords if kw in mvv.lower()]
    services_in_bmc = [kw for kw in service_keywords if kw in bmc.lower()]
    services_in_impact = [kw for kw in service_keywords if kw in impact.lower()]
    
    # Check overlap
    common_services = set(services_in_profile) & (
        set(services_in_mvv) | set(services_in_bmc) | set(services_in_impact)
    )
    
    service_consistent = len(common_services) >= 2
    results["checks_total"] += 1
    if service_consistent:
        results["checks_passed"] += 1
        print(f"  ✓ PASS: Common services found - {common_services}")
    else:
        print(f"  ⚠ WARNING: Service alignment may be weak")
    results["details"]["service_consistency"] = service_consistent
    
    # Check 4: Values reflection in other documents
    print("\n📋 Check 4: Values Reflected in Other Documents")
    
    # Extract values from MVV
    values_section = mvv.lower().split("valores:")[-1] if "valores:" in mvv.lower() else ""
    value_keywords = ["transparência", "inovação", "ética", "sustentabilidade",
                     "compromisso", "excelência", "confiança", "impacto"]
    
    values_in_mvv = [kw for kw in value_keywords if kw in values_section]
    values_in_swot = [kw for kw in value_keywords if kw in swot.lower()]
    values_in_impact = [kw for kw in value_keywords if kw in impact.lower()]
    
    values_reflected = len(set(values_in_mvv) & (set(values_in_swot) | set(values_in_impact))) >= 1
    results["checks_total"] += 1
    if values_reflected:
        results["checks_passed"] += 1
        print(f"  ✓ PASS: Values reflected in other documents")
    else:
        print(f"  ⚠ WARNING: Values may not be well-integrated")
    results["details"]["values_reflection"] = values_reflected
    
    # Overall consistency score
    consistency_score = results["checks_passed"] / results["checks_total"] if results["checks_total"] > 0 else 0
    
    print(f"\n📊 OVERALL CONSISTENCY SCORE: {consistency_score*100:.1f}%")
    print(f"    Checks passed: {results['checks_passed']}/{results['checks_total']}")
    print(f"    Target: ≥75% for acceptable coherence")
    
    if consistency_score >= 0.75:
        print("    ✓ PASS: Documents show good cross-consistency")
    else:
        print("    ⚠ WARNING: Some consistency issues detected")
    
    results["consistency_score"] = consistency_score
    print("="*70)
    
    return results


# Run cross-document validation
consistency_results = validate_cross_document_consistency(
    company_profile['profile_text'],
    mvv_document,
    impact_report_document,
    bmc_document,
    swot_document
)


VALIDATION: Cross-Document Consistency

📋 Check 1: Company Size Consistency
  ✓ PASS: Size indicators found - ['profile: pequena', 'profile: média']

📋 Check 2: Geographic Scope Consistency
  ✓ PASS: Geographic scope mentioned across documents

📋 Check 3: Service/Specialization Alignment
  ✓ PASS: Common services found - {'treinamento', 'esg', 'consultoria', 'carbono', 'sustentabilidade'}

📋 Check 4: Values Reflected in Other Documents
  ✓ PASS: Values reflected in other documents

📊 OVERALL CONSISTENCY SCORE: 100.0%
    Checks passed: 4/4
    Target: ≥75% for acceptable coherence
    ✓ PASS: Documents show good cross-consistency


### 11.2 Structure Preservation Score
Implementing fidelity metrics from SynEval paper (Paper 3)

In [48]:
def validate_structure_preservation(mvv: str, impact: str, bmc: str, swot: str):
    """
    Compute Structure Preservation Score (SPS) from SynEval framework.
    
    Args:
        mvv: Mission/Vision/Values text
        impact: Social Impact Report text
        bmc: Business Model Canvas text
        swot: SWOT Analysis text
    """
    
    print("\n" + "="*70)
    print("VALIDATION: Structure Preservation Score (SPS)")
    print("="*70)
    
    results = {}
    
    # Document 1: MVV
    print("\n📄 Document 1: Mission, Vision & Values")
    mvv_elements = ["missão", "visão", "valores"]
    mvv_present = [e for e in mvv_elements if e in mvv.lower()]
    mvv_sps = len(mvv_present) / len(mvv_elements)
    results['doc1_mvv'] = mvv_sps
    print(f"  Structure Preservation Score: {mvv_sps*100:.1f}%")
    print(f"  Present elements: {mvv_present}")
    
    # Document 3: Impact Report
    print("\n📄 Document 3: Social Impact Report")
    impact_elements = ["impacto", "indicadores", "stakeholder"]
    impact_present = [e for e in impact_elements if e in impact.lower()]
    impact_sps = len(impact_present) / len(impact_elements)
    results['doc3_impact'] = impact_sps
    print(f"  Structure Preservation Score: {impact_sps*100:.1f}%")
    print(f"  Present elements: {impact_present}")
    
    # Document 4: BMC
    print("\n📄 Document 4: Business Model Canvas")
    bmc_elements = ["clientes", "valor", "receita", "recursos", "atividades", 
                    "parcerias", "custos"]
    bmc_present = [e for e in bmc_elements if e in bmc.lower()]
    bmc_sps = len(bmc_present) / len(bmc_elements)
    results['doc4_bmc'] = bmc_sps
    print(f"  Structure Preservation Score: {bmc_sps*100:.1f}%")
    print(f"  Present elements: {bmc_present} ({len(bmc_present)}/{len(bmc_elements)})")
    
    # Document 5: SWOT
    print("\n📄 Document 5: SWOT Analysis")
    swot_elements = ["forças", "fraquezas", "oportunidades", "ameaças"]
    swot_present = [e for e in swot_elements if e in swot.lower()]
    swot_sps = len(swot_present) / len(swot_elements)
    results['doc5_swot'] = swot_sps
    print(f"  Structure Preservation Score: {swot_sps*100:.1f}%")
    print(f"  Present elements: {swot_present}")
    
    # Overall
    overall_sps = sum(results.values()) / len(results)
    print(f"\n📊 Overall Structure Preservation Score: {overall_sps*100:.1f}%")
    print(f"    Target: >90% for high fidelity")
    print("="*70)
    
    return results


# Run validation
validation_results = validate_structure_preservation(
    mvv_document, 
    impact_report_document, 
    bmc_document, 
    swot_document
)


VALIDATION: Structure Preservation Score (SPS)

📄 Document 1: Mission, Vision & Values
  Structure Preservation Score: 100.0%
  Present elements: ['missão', 'visão', 'valores']

📄 Document 3: Social Impact Report
  Structure Preservation Score: 100.0%
  Present elements: ['impacto', 'indicadores', 'stakeholder']

📄 Document 4: Business Model Canvas
  Structure Preservation Score: 100.0%
  Present elements: ['clientes', 'valor', 'receita', 'recursos', 'atividades', 'parcerias', 'custos'] (7/7)

📄 Document 5: SWOT Analysis
  Structure Preservation Score: 100.0%
  Present elements: ['forças', 'fraquezas', 'oportunidades', 'ameaças']

📊 Overall Structure Preservation Score: 100.0%
    Target: >90% for high fidelity


### 11.3 Semantic Similarity Validation

Academic Basis: Sentence-BERT embeddings for fidelity assessment (Reimers & Gurevych, 2019)
This metric validates that synthetic documents are:

Domain-aligned (similar to seed companies)
Not memorized (not too similar - avoiding plagiarism)

In [49]:
def compute_semantic_similarity(
    synthetic_texts: Dict[str, str],
    seed_companies: List[Dict],
    model_name: str = 'paraphrase-multilingual-mpnet-base-v2'
) -> Dict[str, Any]:
    """
    Compute semantic similarity between synthetic documents and seed data.
    
    Validates:
    1. Fidelity: Are synthetic docs domain-aligned with seeds?
    2. Memorization: Are synthetic docs too similar (plagiarism risk)?
    
    Academic basis: Sentence-BERT (Reimers & Gurevych, 2019)
    
    Args:
        synthetic_texts: Dict of {doc_type: text} for synthetic documents
        seed_companies: Seed company data
        model_name: Sentence transformer model
    
    Returns:
        Dictionary with similarity metrics
    """
    
    print("\n" + "="*70)
    print("VALIDATION: Semantic Similarity to Seeds")
    print("="*70)
    print(f"Loading model: {model_name}")
    
    # Load sentence transformer model
    model = SentenceTransformer(model_name)
    print("✓ Model loaded")
    
    results = {}
    
    # ========================================
    # 1. MISSION SIMILARITY
    # ========================================
    print("\n📄 Document 1: Mission Similarity")
    
    # Extract missions from seeds
    seed_missions = [company['mission'] for company in seed_companies]
    
    # Extract synthetic mission
    mvv_text = synthetic_texts.get('mvv', '')
    synthetic_mission = ""
    if "missão:" in mvv_text.lower():
        mission_section = mvv_text.lower().split("missão:")[1]
        if "visão:" in mission_section:
            synthetic_mission = mission_section.split("visão:")[0].strip()
        else:
            synthetic_mission = mission_section[:200].strip()
    
    if synthetic_mission:
        # Encode
        synthetic_emb = model.encode([synthetic_mission])
        seed_embs = model.encode(seed_missions)
        
        # Compute similarities
        similarities = cosine_similarity(synthetic_emb, seed_embs)[0]
        
        mean_sim = np.mean(similarities)
        max_sim = np.max(similarities)
        std_sim = np.std(similarities)
        
        memorization_risk = (max_sim > 0.90)
        
        print(f"  Mean similarity: {mean_sim:.3f}")
        print(f"  Max similarity: {max_sim:.3f}")
        print(f"  Std deviation: {std_sim:.3f}")
        print(f"  Memorization risk (>0.90): {'YES ⚠️' if memorization_risk else 'NO ✓'}")
        
        results['mission'] = {
            'mean_similarity': float(mean_sim),
            'max_similarity': float(max_sim),
            'std_similarity': float(std_sim),
            'memorization_risk': memorization_risk
        }
    else:
        print("  ⚠️ WARNING: Could not extract mission text")
        results['mission'] = None
    
    # ========================================
    # 2. VALUES SIMILARITY
    # ========================================
    print("\n📄 Document 1: Values Similarity")
    
    # Extract values from seeds
    seed_values_texts = [", ".join(company['values']) for company in seed_companies]
    
    # Extract synthetic values
    synthetic_values_text = ""
    if "valores:" in mvv_text.lower():
        values_section = mvv_text.lower().split("valores:")[1]
        synthetic_values_text = values_section[:300].strip()
    
    if synthetic_values_text:
        # Encode
        synthetic_emb = model.encode([synthetic_values_text])
        seed_embs = model.encode(seed_values_texts)
        
        # Compute similarities
        similarities = cosine_similarity(synthetic_emb, seed_embs)[0]
        
        mean_sim = np.mean(similarities)
        max_sim = np.max(similarities)
        std_sim = np.std(similarities)
        
        memorization_risk = (max_sim > 0.90)
        
        print(f"  Mean similarity: {mean_sim:.3f}")
        print(f"  Max similarity: {max_sim:.3f}")
        print(f"  Std deviation: {std_sim:.3f}")
        print(f"  Memorization risk (>0.90): {'YES ⚠️' if memorization_risk else 'NO ✓'}")
        
        results['values'] = {
            'mean_similarity': float(mean_sim),
            'max_similarity': float(max_sim),
            'std_similarity': float(std_sim),
            'memorization_risk': memorization_risk
        }
    else:
        print("  ⚠️ WARNING: Could not extract values text")
        results['values'] = None
    
    # ========================================
    # 3. OVERALL DOCUMENT-LEVEL SIMILARITY
    # ========================================
    print("\n📄 Overall Document Similarity")
    
    # Use full MVV text
    full_synthetic_text = synthetic_texts.get('mvv', '')
    
    # Use full seed company descriptions (mission + vision + values)
    seed_full_texts = []
    for company in seed_companies:
        text = company['mission']
        if company.get('vision'):
            text += " " + company['vision']
        text += " " + ", ".join(company['values'])
        seed_full_texts.append(text)
    
    if full_synthetic_text:
        # Encode
        synthetic_emb = model.encode([full_synthetic_text])
        seed_embs = model.encode(seed_full_texts)
        
        # Compute similarities
        similarities = cosine_similarity(synthetic_emb, seed_embs)[0]
        
        mean_sim = np.mean(similarities)
        max_sim = np.max(similarities)
        std_sim = np.std(similarities)
        
        memorization_risk = (max_sim > 0.90)
        
        print(f"  Mean similarity: {mean_sim:.3f}")
        print(f"  Max similarity: {max_sim:.3f}")
        print(f"  Std deviation: {std_sim:.3f}")
        print(f"  Memorization risk (>0.90): {'YES ⚠️' if memorization_risk else 'NO ✓'}")
        
        results['overall'] = {
            'mean_similarity': float(mean_sim),
            'max_similarity': float(max_sim),
            'std_similarity': float(std_sim),
            'memorization_risk': memorization_risk
        }
    
    # ========================================
    # INTERPRETATION
    # ========================================
    print("\n" + "="*70)
    print("INTERPRETATION:")
    print("="*70)
    
    if results.get('overall'):
        mean_overall = results['overall']['mean_similarity']
        
        if 0.40 <= mean_overall <= 0.70:
            print("✓ EXCELLENT: Domain-aligned but diverse (0.40-0.70 range)")
            interpretation = "excellent"
        elif mean_overall < 0.40:
            print("⚠️ WARNING: Low similarity - may be off-topic (<0.40)")
            interpretation = "low_fidelity"
        elif mean_overall > 0.85:
            print("⚠️ WARNING: High similarity - possible memorization (>0.85)")
            interpretation = "high_memorization"
        else:
            print("✓ GOOD: Acceptable domain alignment (0.70-0.85 range)")
            interpretation = "good"
        
        results['interpretation'] = interpretation
    
    print("="*70)
    
    return results


# Run semantic similarity validation
semantic_similarity_results = compute_semantic_similarity(
    synthetic_texts={
        'mvv': mvv_document,
        'impact': impact_report_document,
        'bmc': bmc_document,
        'swot': swot_document
    },
    seed_companies=seed_companies
)


VALIDATION: Semantic Similarity to Seeds
Loading model: paraphrase-multilingual-mpnet-base-v2
✓ Model loaded

📄 Document 1: Mission Similarity
  Mean similarity: 0.576
  Max similarity: 0.862
  Std deviation: 0.163
  Memorization risk (>0.90): NO ✓

📄 Document 1: Values Similarity
  Mean similarity: 0.601
  Max similarity: 0.764
  Std deviation: 0.120
  Memorization risk (>0.90): NO ✓

📄 Overall Document Similarity
  Mean similarity: 0.673
  Max similarity: 0.864
  Std deviation: 0.110
  Memorization risk (>0.90): NO ✓

INTERPRETATION:
✓ EXCELLENT: Domain-aligned but diverse (0.40-0.70 range)


### 11.4 Human Eval